In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import seaborn as sns
import os

data_path = r"C:\Users\egkbo\dallas_heat_island\data\raw"

income = pd.read_csv(os.path.join(data_path, "ACSDT5Y2023.B19013-Data.csv"), skiprows = 1)
population = pd.read_csv(os.path.join(data_path, "ACSDT5Y2023.B01003-Data.csv"), skiprows=1)
housing = pd.read_csv(os.path.join(data_path, "ACSDT5Y2023.B25034-Data.csv"), skiprows=1)

print("Income shape:", income.shape)
print("Population shape:", population.shape)
print("Housing shape:", housing.shape)

Income shape: (645, 5)
Population shape: (645, 5)
Housing shape: (645, 25)


In [4]:
# load dallas shapefile
shapefile_path = os.path.join(data_path, "tl_2022_48_tract.zip")
dallas_tracts = gpd.read_file(shapefile_path)

# filter to just dallas county (FIPS code 113)
dallas_tracts = dallas_tracts[dallas_tracts['COUNTYFP'] == '113']

print("Dallas tracts shape:", dallas_tracts.shape)
dallas_tracts.head()

Dallas tracts shape: (645, 13)


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
176,48,113,016638,48113016638,166.38,Census Tract 166.38,G5020,S,3449205,17743,+32.6295891,-096.8758118,"POLYGON ((-96.88706 32.64507, -96.88652 32.645..."
264,48,113,013630,48113013630,136.30,Census Tract 136.30,G5020,S,292256,0,+32.9444788,-096.8067075,"POLYGON ((-96.81022 32.94356, -96.81003 32.943..."
841,48,113,012211,48113012211,122.11,Census Tract 122.11,G5020,S,1318895,0,+32.8022212,-096.6890352,"POLYGON ((-96.69814 32.80415, -96.69698 32.805..."
842,48,113,012400,48113012400,124,Census Tract 124,G5020,S,3053580,0,+32.8282285,-096.6848089,"POLYGON ((-96.69574 32.82394, -96.6957 32.824,..."
843,48,113,016513,48113016513,165.13,Census Tract 165.13,G5020,S,5281302,0,+32.6352862,-096.9335437,"POLYGON ((-96.94367 32.61815, -96.94332 32.618..."


In [5]:
# load landsat files
lst_path = os.path.join(data_path, "LC09_L2SP_027035_20230819_20230821_02_T1_ST_B10.TIF")
b4_path = os.path.join(data_path, "LC09_L2SP_027035_20230819_20230821_02_T1_SR_B4.TIF")
b5_path = os.path.join(data_path, "LC09_L2SP_027035_20230819_20230821_02_T1_SR_B5.TIF")

# open and inspect each file
with rasterio.open(lst_path) as src:
    print("LST CRS:", src.crs)
    print("LST shape:", src.shape)
    print("LST bounds:", src.bounds)
    lst_data = src.read(1).astype("float32")

print("\nLST min:", lst_data.min())
print("LST max:", lst_data.max())

LST CRS: EPSG:32614
LST shape: (7691, 7571)
LST bounds: BoundingBox(left=632985.0, bottom=3876285.0, right=860115.0, top=4107015.0)

LST min: 0.0
LST max: 54823.0


In [6]:
# load b4 and b5 bands
with rasterio.open(b4_path) as src:
    b4_data = src.read(1).astype("float32")
    
with rasterio.open(b5_path) as src:
    b5_data = src.read(1).astype("float32")

print("B4 shape:", b4_data.shape)
print("B5 shape:", b5_data.shape)

# load enviroatlas - need to unzip first
import zipfile
envi_zip = os.path.join(data_path, "CONUS_metrics_CSV.zip")
envi_extract = os.path.join(data_path, "enviroatlas")

with zipfile.ZipFile(envi_zip, 'r') as z:
    z.extractall(envi_extract)
    print("\nEnviroAtlas files extracted:")
    for f in z.namelist():
        print(f)

B4 shape: (7691, 7571)
B5 shape: (7691, 7571)

EnviroAtlas files extracted:
CONUS_metrics_CSV/
CONUS_metrics_CSV/CONUS_changelog_08-05-2025.txt
CONUS_metrics_CSV/CONUS_metadata.zip
CONUS_metrics_CSV/CONUS_metrics_CSV/
CONUS_metrics_CSV/CONUS_metrics_CSV/AgBuffers.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/AgW_Demand.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/Ag_On_Slopes.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/Ag_P_Balance.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/AMAD_CONUS.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/AvgPrecip_NHDPv2_WBD.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/BigGameHunting_RecreationDemand.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/Biodiversity.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/BirdWatching_RecreationDemand.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/BNF06_NHDPv2_WBD.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/Business_Vacancy_Rate_Tract10.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/Business_Vacancy_Units_Tract10.csv
CONUS_metrics_CSV/CONUS_metrics_CSV/cbnf06_NHDPv2_WBD.csv
CONUS_metric

In [7]:
# find the relevant enviroatlas files
envi_path = os.path.join(data_path, "enviroatlas", "CONUS_metrics_CSV", "CONUS_metrics_CSV")

# list all files and filter for relevant ones
all_files = os.listdir(envi_path)
keywords = ["imperv", "tree", "canopy", "green", "urban", "land"]

print("Potentially relevant files:")
for f in all_files:
    if any(k in f.lower() for k in keywords):
        print(f)

Potentially relevant files:
Impervious_CONUS.csv
LandCoverInFloodplains_CONUS.csv
LandCover_CONUS.csv
PCanopy_CONUS.csv
PercentPotentialWetlandAreasOnCropland_CONUS.csv
PercentPotentialWetlandAreas_CONUS.csv
RiparianCanopy_CONUS.csv


In [8]:
# load enviroatlas impervious and canopy data
impervious = pd.read_csv(os.path.join(envi_path, "Impervious_CONUS.csv"))
canopy = pd.read_csv(os.path.join(envi_path, "PCanopy_CONUS.csv"))

print("Impervious shape:", impervious.shape)
print("Canopy shape:", canopy.shape)

print("\nImpervious columns:", impervious.columns.tolist())
print("Canopy columns:", canopy.columns.tolist())

print("\nImpervious head:")
print(impervious.head())

Impervious shape: (82915, 3)
Canopy shape: (82915, 3)

Impervious columns: ['OBJECTID', 'HUC_12', 'PIMPV']
Canopy columns: ['OBJECTID', 'HUC_12', 'pCanopy']

Impervious head:
   OBJECTID       HUC_12     PIMPV
0         1  10100020101  0.297222
1         2  10100020102  0.250103
2         3  10100020103  0.216098
3         4  10100020104  0.195608
4         5  10100020105  0.205232


In [9]:
landcover = pd.read_csv(os.path.join(envi_path, "LandCover_CONUS.csv"))
print("LandCover shape:", landcover.shape)
print("LandCover columns:", landcover.columns.tolist())
print(landcover.head())

LandCover shape: (82915, 11)
LandCover columns: ['OBJECTID', 'HUC_12', 'PAGT', 'PAGC', 'PAGP', 'PDEV', 'PFOR', 'PFOR90', 'N_INDEX', 'PWETL95', 'PWETL']
   OBJECTID       HUC_12  PAGT  PAGC  PAGP  PDEV       PFOR     PFOR90  \
0         1  10100020101   0.0   0.0   0.0     2  82.143944  92.373772   
1         2  10100020102   0.0   0.0   0.0     2  79.948639  90.972382   
2         3  10100020103   0.0   0.0   0.0     2  83.035744  95.061989   
3         4  10100020104   0.0   0.0   0.0     2  87.037468  94.330261   
4         5  10100020105   0.0   0.0   0.0     1  83.511375  90.318550   

     N_INDEX   PWETL95      PWETL  
0  97.697563  0.977045  11.206873  
1  98.224152  1.129879  12.153618  
2  98.331970  1.354742  13.380980  
3  98.339500  0.220048   7.512836  
4  98.650886  2.995095   9.802274  


In [10]:
print("=== DATA LOADING SUMMARY ===")
print(f"Income (census tracts):     {income.shape}")
print(f"Population (census tracts): {population.shape}")
print(f"Housing (census tracts):    {housing.shape}")
print(f"Dallas shapefile (tracts):  {dallas_tracts.shape}")
print(f"Landsat LST band:           {lst_data.shape}")
print(f"Landsat B4 band:            {b4_data.shape}")
print(f"Landsat B5 band:            {b5_data.shape}")
print(f"Impervious (watersheds):    {impervious.shape}")
print(f"Canopy (watersheds):        {canopy.shape}")
print("\nAll data loaded successfully!")

=== DATA LOADING SUMMARY ===
Income (census tracts):     (645, 5)
Population (census tracts): (645, 5)
Housing (census tracts):    (645, 25)
Dallas shapefile (tracts):  (645, 13)
Landsat LST band:           (7691, 7571)
Landsat B4 band:            (7691, 7571)
Landsat B5 band:            (7691, 7571)
Impervious (watersheds):    (82915, 3)
Canopy (watersheds):        (82915, 3)

All data loaded successfully!
